# Prediction-interval calibration: gap to the best competing method

This D9 (uncertainty calibration) study compares each mboostrs method with the
narrowest coverage-eligible competing interval method. The exact oracle interval is shown as a secondary reference.

The protocol is in `interval_calibration/PROTOCOL.md`. The results are in
`interval_calibration/results.json`, and the gap table is produced by
`interval_calibration/gap_to_best.py`.

## Two tracks

- **Known specification.** mboostrs is given the true mean structure and the
  Gaussian family. Methods: `mboostrs_split`, `mboostrs_cqr`, `mboostrs_cv+`.
- **Discovery.** mboostrs receives only `(y, x1, x2)` and calls `mb.discover`,
  which chooses the formula and spread model. Method: `mboostrs_discover`.

Competitors receive only `(y, x1, x2)` in both tracks: CatBoost CQR, CatBoost
split conformal, scikit-learn gradient-boosting CQR, and LightGBM CQR and split.
Three planted processes: homoscedastic normal, heteroscedastic normal, and
heavy-tailed t(3). 20 seeds each, 1000 training and 2000 test rows.

In [1]:
import json
from pathlib import Path

import pandas as pd

root = Path("interval_calibration")
gaps = json.loads((root / "gap_to_best.json").read_text())
summary = json.loads((root / "summary.json").read_text())

## Gap to the best competing method

Each row is one mboostrs method on one process. The best competitor is the one
with the smallest mean width among competitors whose coverage is within 0.02 of
0.90, chosen on test widths (a retrospective best). The gap is the paired
per-seed width ratio minus one, in percent. Positive means mboostrs is wider.

In [2]:
rows = []
for process, block in gaps.items():
    for method, m in block["methods"].items():
        rows.append({
            "process": process,
            "track": m["track"].replace("_", " "),
            "method": method,
            "coverage": round(m["coverage"], 3),
            "best competitor": block["best_competitor"],
            "gap to best (%)": round(m["gap_to_best_competitor_pct"], 1),
            "gap s.e. (%)": round(m["gap_se_pct"], 1),
            "gap to oracle (%)": round(m["gap_to_oracle_pct"], 1),
        })
gap_table = pd.DataFrame(rows)
gap_table

,process,track,method,coverage,best competitor,gap to best (%),gap s.e. (%),gap to oracle (%)
0,homoscedastic,known specification,mboostrs_split,0.902,catboost_split,-1.4,0.9,7.1
1,homoscedastic,known specification,mboostrs_cqr,0.902,catboost_split,-0.9,1.0,7.7
2,homoscedastic,known specification,mboostrs_cv+,0.903,catboost_split,-1.1,0.8,7.4
3,homoscedastic,discovery,mboostrs_discover,0.900,catboost_split,-6.5,0.7,1.7
4,heteroscedastic,known specification,mboostrs_split,0.899,lightgbm_cqr,6.3,1.4,15.6
5,heteroscedastic,known specification,mboostrs_cqr,0.903,lightgbm_cqr,-1.9,0.8,6.7
6,heteroscedastic,known specification,mboostrs_cv+,0.899,lightgbm_cqr,5.6,1.0,14.8
7,heteroscedastic,discovery,mboostrs_discover,0.897,lightgbm_cqr,2.6,1.2,11.6
8,heavy_tailed_t3,known specification,mboostrs_split,0.902,catboost_split,-1.3,1.4,9.6
9,heavy_tailed_t3,known specification,mboostrs_cqr,0.904,catboost_split,4.9,1.3,16.7


## Competitor widths

Mean interval width of every competing method, per process. Lower is narrower,
at the same coverage. The gap table above uses the smallest width in each block.

In [3]:
width = {}
for process, block in summary["dgps"].items():
    width[process] = {m: round(s["mean_width"], 3) for m, s in block["methods"].items()}
width_table = pd.DataFrame(width).T
width_table

,catboost_cqr,catboost_split,lightgbm_cqr,lightgbm_split,mboostrs_cqr,mboostrs_cv+,mboostrs_discover,mboostrs_split,oracle,sklearn_gbr_cqr
homoscedastic,3.836,3.577,3.663,3.696,3.541,3.533,3.344,3.524,3.290,3.691
heteroscedastic,3.934,4.184,3.747,4.264,3.675,3.952,3.840,3.979,3.443,3.841
heavy_tailed_t3,3.427,3.028,3.224,3.175,3.170,2.958,2.785,2.979,2.717,3.294


## Reading the gaps

- **Known specification, homoscedastic and heavy-tailed.** The split and CV+
  methods are narrower than the best competitor, by about 1 to 2%.
- **Known specification, heteroscedastic.** CQR is narrower by 1.9%. Split and
  CV+ are wider by 5.6 to 6.3%; they under-cover the high-scale third
  (issue #543).
- **Known specification, heavy-tailed.** CQR is wider by 4.9% than CatBoost
  split conformal.
- **Discovery.** Narrower than the best competitor on homoscedastic and
  heavy-tailed data (6.5 to 7.8%), and wider by 2.6% on heteroscedastic data.

The discovery track fits on all training rows, and the conformal methods split
them. The two tracks differ in data use as well as in specification.

The competitors are fixed, untuned settings. The best competitor changes by
process, so the gap is not a single figure for the whole benchmark.